# Inverse Folding (Ubiquitin)

**BioPipelines example** — inverse folding of ubiquitin using ProteinMPNN. Designed sequences are refolded with AlphaFold2 and filtered by RMSD and pLDDT. Sequences passing the filter are codon-optimised for *E. coli* expression with DNAEncoder.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e .
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba
!micromamba create -f environments/biopipelines.yaml -y

Cloning into 'biopipelines'...
remote: Enumerating objects: 7678, done.
remote: Counting objects: 100% (1048/1048), done.
remote: Compressing objects: 100% (390/390), done.
remote: Total 7678 (delta 717), reused 877 (delta 645), pack-reused 6630 (from 1)
Receiving objects: 100% (7678/7678), 16.91 MiB | 17.24 MiB/s, done.
Resolving deltas: 100% (5872/5872), done.
/content/biopipelines
Obtaining file:///content/biopipelines
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 14.0 MB/s eta 0:00:00
  Building editable for biopipelines (pyproject.toml) ... done
  Created wheel for biopipelines: filename=biopipelines-1.1.1-0.e

In [2]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines.rfdiffusion import RFdiffusion
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold
from biopipelines.pymol import PyMOL

with Pipeline(project="Setup", job="InstallTools"):
    ProteinMPNN.install()
    AlphaFold.install()
    PyMOL.install() # For structure-based analysis like ConformationalChange


Running ProteinMPNN_installation (step 1)
=== Activating Environment ===
Requested: biopipelines
Environment: biopipelines
Location: /root/.local/share/mamba/envs/biopipelines
Python: /root/.local/share/mamba/envs/biopipelines/bin/python
Python version: Python 3.12.13
=== Installing ProteinMPNN ===
Cloning into 'ProteinMPNN'...
Shared env 'SE3nv' missing — creating it from the BioPipelines spec.
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                           ⧖ Starting
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                  

## Cell 3: Inverse Folding Pipeline

Fetches ubiquitin (PDB: 4LCD, chain E) and generates 10 sequences with ProteinMPNN (soluble model).
AlphaFold2 refolds each sequence; those with RMSD < 1.5 Å and pLDDT > 80 are codon-optimised for *E. coli* expression.

In [3]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold
from biopipelines.conformational_change import ConformationalChange
from biopipelines.panda import Panda
from biopipelines.dna_encoder import DNAEncoder

with Pipeline(project="Ubiquitin", job="InverseFolding"):
    ubiquitin = PDB("4LCD", chain="E")
    sequences = ProteinMPNN(structures=ubiquitin,
                            num_sequences=10,
                            soluble_model=True)
    folded = AlphaFold(proteins=sequences)
    dna = DNAEncoder(sequences=sequences,
                     organism="EC")
    conf_change = ConformationalChange(reference_structures=ubiquitin,
                                       target_structures=folded)
    filtered_sequences = Panda(tables=[folded.tables.confidence,
                                       conf_change.tables.changes],
                               operations=[Panda.merge(),
                                            Panda.filter("RMSD < 1.5 and plddt > 80")],
                               pool=sequences)
    dna = DNAEncoder(sequences=filtered_sequences,
                     organism="EC")
dna

  4LCD not found locally, will download from RCSB

Running PDB (step 1)
=== Activating Environment ===
Requested: biopipelines
Environment: biopipelines
Location: /root/.local/share/mamba/envs/biopipelines
Python: /root/.local/share/mamba/envs/biopipelines/bin/python
Python version: Python 3.12.13
Fetching 1 structures
Convert: keep as-is (pdb|cif)
PDB IDs: 4LCD
Custom IDs: 4LCD
Priority: pdbs/ -> RCSB download
Output folder: /content/biopipelines/outputs/Ubiquitin/InverseFolding_001/001_PDB
Fetching 1 structures (keep as-is (pdb|cif))
Priority: pdbs/ -> RCSB download
Water molecules will be removed

[1/1] Processing 4LCD -> 4LCD
4LCD not found locally, downloading from RCSB
Cached to pdbs/ folder: /content/biopipelines/pdbs/4LCD.pdb
Successfully downloaded 4LCD as 4LCD: 779382 bytes (rcsb_download (PDB))

Successful fetches saved: /content/biopipelines/outputs/Ubiquitin/InverseFolding_001/001_PDB/structures/structures.csv (1 structures)
Sequences saved: /content/biopipelines/outputs/U

StandardizedOutput({'sequences': DataStream(name='sequences', format='dna', items=10, files=0, map_table=set), 'tables': {'dna': TableInfo(name='dna', path='/content/biopipelines/outputs/Ubiquitin/InverseFolding_001/007_DNAEncoder/tables/dna.csv', columns=['id', 'protein_sequence', 'dna_sequence', 'organism', 'method'], count=1)}, 'output_folder': '/content/biopipelines/outputs/Ubiquitin/InverseFolding_001/007_DNAEncoder', 'excel': '/content/biopipelines/outputs/Ubiquitin/InverseFolding_001/007_DNAEncoder/_extras/dna_sequences.xlsx', 'info': '/content/biopipelines/outputs/Ubiquitin/InverseFolding_001/007_DNAEncoder/_extras/dna_info.txt'})